In [3]:
import pandas as pd
pd.set_option('display.max_columns', None)
import numpy as np
import os
import cmlreaders as cml

In [4]:
def build_gt_events_csv(experiment='ltpFR2', output_path=None):
    """Build a ground-truth events CSV from CML for all subjects/sessions.

    Includes WORD, REC_WORD, and REC_WORD_VV (vocalizations marked with '<>').
    """
    from cmlreaders import CMLReader, get_data_index

    df = get_data_index()
    exp_df = df[df['experiment'] == experiment]

    all_frames = []
    for _, row in exp_df.iterrows():
        subject = row['subject']
        session = row['session']
        montage = row.get('montage', 0)
        localization = row.get('localization', 0)
        try:
            reader = CMLReader(subject, experiment, session,
                               montage=montage, localization=localization)
            evs = reader.load('events')
            filtered = evs.query("type == 'WORD' or type == 'REC_WORD' or type == 'REC_WORD_VV'")
            if len(filtered) > 0:
                all_frames.append(filtered)
        except Exception as e:
            print(f"WARNING: Failed to load events for {subject} session {session}: {e}")
            continue

    if not all_frames:
        raise RuntimeError(f"No events loaded for experiment {experiment}")

    result = pd.concat(all_frames, ignore_index=True)
    print(f"Built GT events: {len(result)} rows, "
          f"{result['subject'].nunique()} subjects, "
          f"{experiment}")
    print(f"  WORD: {(result['type']=='WORD').sum()}, "
          f"REC_WORD: {(result['type']=='REC_WORD').sum()}, "
          f"REC_WORD_VV: {(result['type']=='REC_WORD_VV').sum()}")

    if output_path:
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        result.to_csv(output_path, index=False)
        print(f"Saved to {output_path}")

    return result

In [8]:
## ltpFR2 sessions sorted by vocalizations (with intrusions)

GT_CSV = 'intrusion_exploration/ltpFR2_word_rec_word_vv_events.csv'
gt_df = pd.read_csv(GT_CSV) if os.path.exists(GT_CSV) else build_gt_events_csv(experiment='ltpFR2', output_path=GT_CSV)

rec_evs = gt_df.query("type == 'REC_WORD'")
vv = gt_df.query("type in ('REC_WORD', 'REC_WORD_VV')").copy()
vv['prev_rectime'] = vv.groupby(['subject', 'session'])['rectime'].shift(1)
ext_mask = (vv['rectime'] - vv['prev_rectime'] == 1000)

vv['is_vocalization'] = ~ext_mask & (vv['type'] == 'REC_WORD_VV')
vv['is_extension'] = ext_mask & (vv['type'] == 'REC_WORD_VV')
vv['is_intrusion'] = vv.index.isin(rec_evs[rec_evs['intrusion'] != 0].index)

session_summary = (
    vv.groupby(['subject', 'session'])
    .agg(
        n_recall_events=('type', 'size'),
        n_vocalizations=('is_vocalization', 'sum'),
        n_extensions=('is_extension', 'sum'),
        n_intrusions=('is_intrusion', 'sum'),
    )
    .reset_index()
)
session_summary['vocal_pct'] = (session_summary['n_vocalizations'] / session_summary['n_recall_events'] * 100).round(1)
session_summary['intr_pct'] = (session_summary['n_intrusions'] / session_summary['n_recall_events'] * 100).round(1)

sorted_summary = session_summary.sort_values('n_vocalizations', ascending=False).reset_index()

In [9]:
sorted_summary.to_csv('intrusion_exploration/ltpfr2_summary_sorted.csv', index=False)

In [16]:
idx_df = cml.get_data_index()
idx_df.query("experiment == 'FR1'").subject.unique()

<StringArray>
['R1001P', 'R1002P', 'R1003P', 'R1006P', 'R1010J', 'R1015J', 'R1018P',
 'R1020J', 'R1022J', 'R1023J',
 ...
 'R1462M', 'R1463E', 'R1466J', 'R1467M', 'R1542J', 'R1565T', 'R1569T',
 'R1571T', 'R1572T', 'R1573T']
Length: 283, dtype: str

In [3]:
def ratio_rows(first_pd, second_pd):
    return first_pd.shape[0] / second_pd.shape[0]

In [9]:
import cmlreaders as cml
idx_df = cml.get_data_index()
experiments = idx_df['experiment'].unique()
rows = []

for exp in experiments:
    try:
        GT_CSV = f'intrusion_exploration/{exp}_word_rec_word_vv_events.csv'
        gt_df = pd.read_csv(GT_CSV) if os.path.exists(GT_CSV) else build_gt_events_csv(experiment=exp, output_path=GT_CSV)

        rec_evs = gt_df.query("type == 'REC_WORD'")
        vv = gt_df.query("type == 'REC_WORD' or type == 'REC_WORD_VV'").copy()
        vv['prev_rectime'] = vv.groupby(['subject', 'session'])['rectime'].shift(1)
        ext_mask = (vv['rectime'] - vv['prev_rectime'] == 1000)
        vocalizations = vv[~ext_mask & (vv['type'] == 'REC_WORD_VV')]
        word_extensions = vv[ext_mask & (vv['type'] == 'REC_WORD_VV')]
        intrusions = rec_evs.query("intrusion != 0")

        n_vv = len(vv)
        n_vocal = len(vocalizations)
        n_ext = len(word_extensions)
        n_intr = len(intrusions)

        rows.append({
            'experiment': exp,
            'n_recall_events': n_vv,
            'n_vocalizations': n_vocal,
            'vocal_pct': n_vocal / n_vv * 100 if n_vv else 0,
            'n_word_extensions': n_ext,
            'ext_pct': n_ext / n_vv * 100 if n_vv else 0,
            'n_intrusions': n_intr,
            'intr_pct': n_intr / n_vv * 100 if n_vv else 0,
        })
        print({
            'experiment': exp,
            'n_recall_events': n_vv,
            'n_vocalizations': n_vocal,
            'vocal_pct': n_vocal / n_vv * 100 if n_vv else 0,
            'n_word_extensions': n_ext,
            'ext_pct': n_ext / n_vv * 100 if n_vv else 0,
            'n_intrusions': n_intr,
            'intr_pct': n_intr / n_vv * 100 if n_vv else 0,
        })
    except Exception as e:
        print(f"ERROR [{exp}]: {e}")

summary_df = pd.DataFrame(rows)

# append averages row
sum_row = summary_df[['n_recall_events', 'n_vocalizations', 'n_word_extensions', 'n_intrusions']].sum()
avg = summary_df[['vocal_pct', 'ext_pct', 'intr_pct']].mean()
avg_row = {'experiment': 'AVERAGE', **avg.to_dict(), **sum_row.to_dict()}
ltp_exp = idx_df.query("subject.str.contains('LTP')", engine='python').experiment.unique()
r_exp = idx_df.query("not subject.str.contains('LTP')", engine='python').experiment.unique()
ltp_sum_row = summary_df.query("experiment in @ltp_exp")[['n_recall_events', 'n_vocalizations', 'n_word_extensions', 'n_intrusions']].sum()
ltp_avg = summary_df.query("experiment in @ltp_exp")[['vocal_pct', 'ext_pct', 'intr_pct']].mean()
ltp_avg_row = {'experiment': 'LTP_AVERAGE', **ltp_avg.to_dict(), **ltp_sum_row.to_dict()}

r_sum_row = summary_df.query("experiment in @r_exp")[['n_recall_events', 'n_vocalizations', 'n_word_extensions', 'n_intrusions']].sum()
r_avg = summary_df.query("experiment in @r_exp")[['vocal_pct', 'ext_pct', 'intr_pct']].mean()
r_avg_row = {'experiment': 'R_AVERAGE', **r_avg.to_dict(), **r_sum_row.to_dict()}

summary_df = pd.concat([summary_df, pd.DataFrame([avg_row])], ignore_index=True)
summary_df = pd.concat([summary_df, pd.DataFrame([ltp_avg_row])], ignore_index=True)
summary_df = pd.concat([summary_df, pd.DataFrame([r_avg_row])], ignore_index=True)

summary_df.to_csv('intrusion_exploration/word_rec_vv_summary.csv', index=False)
print("Saved to intrusion_exploration/word_rec_vv_summary.csv\n")
summary_df

{'experiment': 'ValueCourier', 'n_recall_events': 1760, 'n_vocalizations': 58, 'vocal_pct': 3.295454545454545, 'n_word_extensions': 190, 'ext_pct': 10.795454545454545, 'n_intrusions': 120, 'intr_pct': 6.8181818181818175}


/tmp/ipykernel_23340/2552654200.py:9: DtypeWarning: Columns (0: test) have mixed types. Specify dtype option on import or set low_memory=False.
  gt_df = pd.read_csv(GT_CSV) if os.path.exists(GT_CSV) else build_gt_events_csv(experiment=exp, output_path=GT_CSV)


{'experiment': 'ltpFR', 'n_recall_events': 635642, 'n_vocalizations': 17419, 'vocal_pct': 2.7403790183782695, 'n_word_extensions': 12088, 'ext_pct': 1.9016993842445904, 'n_intrusions': 38793, 'intr_pct': 6.102963617885539}
{'experiment': 'ltpFR2', 'n_recall_events': 762691, 'n_vocalizations': 10913, 'vocal_pct': 1.4308546973807217, 'n_word_extensions': 12282, 'ext_pct': 1.6103507187052162, 'n_intrusions': 28865, 'intr_pct': 3.7846257527622584}
{'experiment': 'VFFR', 'n_recall_events': 268071, 'n_vocalizations': 853, 'vocal_pct': 0.3181992830257656, 'n_word_extensions': 72, 'ext_pct': 0.026858556128786775, 'n_intrusions': 4183, 'intr_pct': 1.560407503982154}
{'experiment': 'ltpRepFR', 'n_recall_events': 34910, 'n_vocalizations': 829, 'vocal_pct': 2.3746777427671155, 'n_word_extensions': 36, 'ext_pct': 0.1031223145230593, 'n_intrusions': 15138, 'intr_pct': 43.362933256946434}


/tmp/ipykernel_23340/2552654200.py:9: DtypeWarning: Columns (0: distractor) have mixed types. Specify dtype option on import or set low_memory=False.
  gt_df = pd.read_csv(GT_CSV) if os.path.exists(GT_CSV) else build_gt_events_csv(experiment=exp, output_path=GT_CSV)


{'experiment': 'NiclsCourierClosedLoop', 'n_recall_events': 13401, 'n_vocalizations': 270, 'vocal_pct': 2.0147750167897915, 'n_word_extensions': 382, 'ext_pct': 2.8505335422729647, 'n_intrusions': 1001, 'intr_pct': 7.469591821505858}
{'experiment': 'NiclsCourierReadOnly', 'n_recall_events': 28448, 'n_vocalizations': 850, 'vocal_pct': 2.987907761529809, 'n_word_extensions': 1286, 'ext_pct': 4.5205286839145105, 'n_intrusions': 3203, 'intr_pct': 11.25913948256468}


...

ERROR [PS3]: No events loaded for experiment PS3
ERROR [PS2]: No events loaded for experiment PS2
ERROR [TH1]: No events loaded for experiment TH1
{'experiment': 'FR3', 'n_recall_events': 3776, 'n_vocalizations': 651, 'vocal_pct': 17.240466101694913, 'n_word_extensions': 122, 'ext_pct': 3.2309322033898304, 'n_intrusions': 376, 'intr_pct': 9.957627118644067}
ERROR [PS2.1]: No events loaded for experiment PS2.1
ERROR [PAL3]: No events loaded for experiment PAL3
ERROR [TH3]: No events loaded for experiment TH3
 /protocols/r1/subjects/R1556J/experiments/OPS/sessions/0/behavioral/current_processed/all_events.json
/data/events/pyFR/R1556J_1_events.mat
ERROR [OPS]: No events loaded for experiment OPS
{'experiment': 'RepFR1', 'n_recall_events': 16692, 'n_vocalizations': 2633, 'vocal_pct': 15.774023484303859, 'n_word_extensions': 796, 'ext_pct': 4.768751497723461, 'n_intrusions': 2207, 'intr_pct': 13.221902707884018}
{'experiment': 'catFR3', 'n_recall_events': 1778, 'n_vocalizations': 211, 'voc

/tmp/ipykernel_23340/2552654200.py:9: DtypeWarning: Columns (0: is_stim, 1: stim_list, 2: distractor) have mixed types. Specify dtype option on import or set low_memory=False.
  gt_df = pd.read_csv(GT_CSV) if os.path.exists(GT_CSV) else build_gt_events_csv(experiment=exp, output_path=GT_CSV)


{'experiment': 'FR5', 'n_recall_events': 2586, 'n_vocalizations': 200, 'vocal_pct': 7.733952049497293, 'n_word_extensions': 31, 'ext_pct': 1.1987625676720803, 'n_intrusions': 1057, 'intr_pct': 40.8739365815932}
{'experiment': 'PS4_catFR', 'n_recall_events': 621, 'n_vocalizations': 15, 'vocal_pct': 2.4154589371980677, 'n_word_extensions': 7, 'ext_pct': 1.1272141706924315, 'n_intrusions': 61, 'intr_pct': 9.822866344605476}
ERROR [THR]: No events loaded for experiment THR
{'experiment': 'PS4_FR', 'n_recall_events': 687, 'n_vocalizations': 28, 'vocal_pct': 4.075691411935954, 'n_word_extensions': 3, 'ext_pct': 0.43668122270742354, 'n_intrusions': 158, 'intr_pct': 22.99854439592431}
ERROR [PAL5]: No events loaded for experiment PAL5
ERROR [THR1]: No events loaded for experiment THR1
{'experiment': 'catFR5', 'n_recall_events': 6711, 'n_vocalizations': 914, 'vocal_pct': 13.619430785277903, 'n_word_extensions': 415, 'ext_pct': 6.183877216510207, 'n_intrusions': 851, 'intr_pct': 12.6806735210847

/tmp/ipykernel_23340/2552654200.py:9: DtypeWarning: Columns (0: lfpfile, 1: timesfile) have mixed types. Specify dtype option on import or set low_memory=False.
  gt_df = pd.read_csv(GT_CSV) if os.path.exists(GT_CSV) else build_gt_events_csv(experiment=exp, output_path=GT_CSV)


,experiment,n_recall_events,n_vocalizations,vocal_pct,n_word_extensions,ext_pct,n_intrusions,intr_pct
0,ValueCourier,1760,58,3.295455,190,10.795455,120,6.818182
1,ltpFR,635642,17419,2.740379,12088,1.901699,38793,6.102964
2,ltpFR2,762691,10913,1.430855,12282,1.610351,28865,3.784626
3,VFFR,268071,853,0.318199,72,0.026859,4183,1.560408
4,ltpRepFR,34910,829,2.374678,36,0.103122,15138,43.362933
5,NiclsCourierClosedLoop,13401,270,2.014775,382,2.850534,1001,7.469592
6,NiclsCourierReadOnly,28448,850,2.987908,1286,4.520529,3203,11.259139
7,ltpDelayRepFRReadOnly,48759,433,0.888041,1423,2.918436,1208,2.477491
8,CourierReinstate1,22872,462,2.019937,616,2.693249,2042,8.927947
9,VCBehOnly,4183,193,4.613913,107,2.557973,838,20.033469


In [14]:
summary_df.sort_values('ext_pct', ascending=False)

,experiment,n_recall_events,n_vocalizations,vocal_pct,n_word_extensions,ext_pct,n_intrusions,intr_pct
12,EFRCourierReadOnly,1077,498,46.239554,229,21.262767,99,9.192201
34,DBOY1,1997,349,17.476214,363,18.177266,141,7.060591
11,EFRCourierOpenLoop,858,253,29.487179,137,15.967366,99,11.538462
33,TICL_catFR,2420,678,28.016529,267,11.033058,368,15.206612
0,ValueCourier,1760,58,3.295455,190,10.795455,120,6.818182
36,pyFR,40047,8048,20.096387,3071,7.668490,8807,21.991660
17,ICatFR1,13758,3807,27.671173,1050,7.631923,1585,11.520570
19,IFR1,9196,2258,24.554154,615,6.687690,1432,15.571988
18,ICatFR6,1388,305,21.974063,87,6.268012,142,10.230548
27,catFR5,6711,914,13.619431,415,6.183877,851,12.680674


In [ ]:
## ── Compare VocalizationClassifier output vs Ground Truth ──────────────────
# For the top 200 ltpFR2 sessions, compare our classifier's vocalization
# detection against the human-annotated ground truth (REC_WORD_VV events).
#
# Matching strategy: for each GT recall event in a trial, find the nearest
# classifier output event by onset time (within 2s tolerance). Then compare
# whether both agree on vocalization vs non-vocalization.

import glob, json

PRED_ROOT = 'results/whisperx-claude-ltpfr2/data/eeg/scalp/ltp/ltpFR2'
GT_CSV = 'intrusion_exploration/ltpFR2_word_rec_word_vv_events.csv'
SUMMARY_CSV = 'intrusion_exploration/ltpfr2_summary_sorted.csv'
ONSET_TOL_MS = 2000  # tolerance for matching events by onset

# Load ground truth (only recall events)
gt_all = pd.read_csv(GT_CSV, usecols=[
    'subject', 'session', 'type', 'item_name', 'rectime', 'intrusion', 'trial'
])
gt_rec = gt_all[gt_all['type'].isin(['REC_WORD', 'REC_WORD_VV'])].copy()

# Load session list
top200 = pd.read_csv(SUMMARY_CSV).head(200)

# ── Per-event matching ──────────────────────────────────────────────────────
rows = []  # one row per matched GT event

for _, sess_row in top200.iterrows():
    subject = sess_row['subject']
    session = int(sess_row['session'])
    
    sess_gt = gt_rec[(gt_rec.subject == subject) & (gt_rec.session == session)]
    pred_dir = os.path.join(PRED_ROOT, subject, f'session_{session}', 'whisperx_out')
    
    if not os.path.isdir(pred_dir):
        continue
    
    for trial_num in sess_gt['trial'].unique():
        trial_gt = sess_gt[sess_gt.trial == trial_num].sort_values('rectime')
        csv_num = int(trial_num) - 1  # trial is 1-indexed, CSV is 0-indexed
        csv_path = os.path.join(pred_dir, f'{csv_num}.csv')
        
        if not os.path.exists(csv_path):
            continue
        
        try:
            pred_df = pd.read_csv(csv_path)
        except Exception:
            continue
        
        if pred_df.empty:
            # No predictions — all GT events are false negatives
            for _, gt_ev in trial_gt.iterrows():
                gt_is_vv = gt_ev['type'] == 'REC_WORD_VV'
                gt_is_intr = (gt_ev['type'] == 'REC_WORD') and (gt_ev['intrusion'] != 0)
                rows.append({
                    'subject': subject, 'session': session, 'trial': trial_num,
                    'gt_type': gt_ev['type'], 'gt_word': gt_ev['item_name'],
                    'gt_rectime': gt_ev['rectime'], 'gt_intrusion': gt_ev['intrusion'],
                    'gt_is_vocalization': gt_is_vv,
                    'gt_is_intrusion': gt_is_intr,
                    'pred_word': None, 'pred_type': None, 'pred_onset': None,
                    'matched': False,
                })
            continue
        
        # Filter out Extension rows from predictions (they're derived, not primary)
        pred_primary = pred_df[pred_df.get('Type', pd.Series(dtype=str)).fillna('') != 'Extension'].copy()
        pred_onsets = pred_primary['Onset'].values
        
        for _, gt_ev in trial_gt.iterrows():
            gt_is_vv = gt_ev['type'] == 'REC_WORD_VV'
            gt_is_intr = (gt_ev['type'] == 'REC_WORD') and (gt_ev['intrusion'] != 0)
            rectime = gt_ev['rectime']
            
            # Find nearest prediction by onset
            if len(pred_onsets) > 0:
                diffs = np.abs(pred_onsets - rectime)
                best_idx = np.argmin(diffs)
                best_diff = diffs[best_idx]
            else:
                best_diff = np.inf
            
            if best_diff <= ONSET_TOL_MS:
                pred_row = pred_primary.iloc[best_idx]
                pred_word = str(pred_row['Word']).strip()
                pred_type = str(pred_row.get('Type', '')).strip()
                pred_is_vv = pred_word == '<>' or pred_type in ('Filler', 'Sentence')
                pred_is_intr = pred_type == 'Intrusion'
                
                rows.append({
                    'subject': subject, 'session': session, 'trial': trial_num,
                    'gt_type': gt_ev['type'], 'gt_word': gt_ev['item_name'],
                    'gt_rectime': rectime, 'gt_intrusion': gt_ev['intrusion'],
                    'gt_is_vocalization': gt_is_vv,
                    'gt_is_intrusion': gt_is_intr,
                    'pred_word': pred_word, 'pred_type': pred_type if pred_type else 'Guess',
                    'pred_onset': int(pred_row['Onset']),
                    'pred_is_vocalization': pred_is_vv,
                    'pred_is_intrusion': pred_is_intr,
                    'onset_diff_ms': best_diff,
                    'matched': True,
                })
            else:
                rows.append({
                    'subject': subject, 'session': session, 'trial': trial_num,
                    'gt_type': gt_ev['type'], 'gt_word': gt_ev['item_name'],
                    'gt_rectime': rectime, 'gt_intrusion': gt_ev['intrusion'],
                    'gt_is_vocalization': gt_is_vv,
                    'gt_is_intrusion': gt_is_intr,
                    'pred_word': None, 'pred_type': None, 'pred_onset': None,
                    'matched': False,
                })

comparison_df = pd.DataFrame(rows)
print(f"Total GT events compared: {len(comparison_df)}")
print(f"Matched events: {comparison_df['matched'].sum()} ({comparison_df['matched'].mean()*100:.1f}%)")
print(f"Unmatched GT events: {(~comparison_df['matched']).sum()}")

In [ ]:
## ── Vocalization Detection: Accuracy & Summary Stats ───────────────────────

matched = comparison_df[comparison_df['matched']].copy()

# ── 1. Vocalization binary classification (GT VV vs our Filler/Sentence) ────
gt_vv = matched['gt_is_vocalization']
pred_vv = matched['pred_is_vocalization']

tp = (gt_vv & pred_vv).sum()
fp = (~gt_vv & pred_vv).sum()
fn = (gt_vv & ~pred_vv).sum()
tn = (~gt_vv & ~pred_vv).sum()

precision = tp / (tp + fp) if (tp + fp) else 0
recall = tp / (tp + fn) if (tp + fn) else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
accuracy = (tp + tn) / len(matched) if len(matched) else 0

print("=" * 60)
print("VOCALIZATION DETECTION (binary: vocalization vs non-vocalization)")
print("=" * 60)
print(f"  True Positives  (GT=VV, Pred=VV):    {tp:5d}")
print(f"  False Positives (GT=word, Pred=VV):   {fp:5d}")
print(f"  False Negatives (GT=VV, Pred=word):   {fn:5d}")
print(f"  True Negatives  (GT=word, Pred=word): {tn:5d}")
print(f"  ─────────────────────────────────────")
print(f"  Accuracy:   {accuracy:.3f}")
print(f"  Precision:  {precision:.3f}")
print(f"  Recall:     {recall:.3f}")
print(f"  F1 Score:   {f1:.3f}")

# ── 2. Confusion matrix ────────────────────────────────────────────────────
print(f"\n{'Confusion Matrix':^40}")
print(f"{'':15s} {'Pred=Vocal':>12s} {'Pred=Word':>12s}")
print(f"{'GT=Vocal':15s} {tp:12d} {fn:12d}")
print(f"{'GT=Word':15s} {fp:12d} {tn:12d}")

# ── 3. Type distribution in predictions (matched events only) ──────────────
print(f"\n{'=' * 60}")
print("PREDICTED TYPE DISTRIBUTION (matched events)")
print("=" * 60)
type_counts = matched['pred_type'].value_counts()
for t, c in type_counts.items():
    print(f"  {t:20s}: {c:5d} ({c/len(matched)*100:.1f}%)")

# ── 4. GT type distribution ────────────────────────────────────────────────
print(f"\n{'=' * 60}")
print("GROUND TRUTH TYPE DISTRIBUTION")
print("=" * 60)
gt_type_counts = matched['gt_type'].value_counts()
for t, c in gt_type_counts.items():
    print(f"  {t:20s}: {c:5d} ({c/len(matched)*100:.1f}%)")

gt_intr_count = matched['gt_is_intrusion'].sum()
print(f"  {'GT intrusions':20s}: {gt_intr_count:5d} ({gt_intr_count/len(matched)*100:.1f}%)")

# ── 5. Intrusion detection ─────────────────────────────────────────────────
print(f"\n{'=' * 60}")
print("INTRUSION DETECTION")
print("=" * 60)
gt_intr = matched['gt_is_intrusion']
pred_intr = matched['pred_is_intrusion']

intr_tp = (gt_intr & pred_intr).sum()
intr_fp = (~gt_intr & pred_intr).sum()
intr_fn = (gt_intr & ~pred_intr).sum()

intr_prec = intr_tp / (intr_tp + intr_fp) if (intr_tp + intr_fp) else 0
intr_rec = intr_tp / (intr_tp + intr_fn) if (intr_tp + intr_fn) else 0
intr_f1 = 2 * intr_prec * intr_rec / (intr_prec + intr_rec) if (intr_prec + intr_rec) else 0

print(f"  TP (GT=intr, Pred=intr):  {intr_tp:5d}")
print(f"  FP (GT≠intr, Pred=intr):  {intr_fp:5d}")
print(f"  FN (GT=intr, Pred≠intr):  {intr_fn:5d}")
print(f"  Precision: {intr_prec:.3f}   Recall: {intr_rec:.3f}   F1: {intr_f1:.3f}")

# ── 6. Cross-tab: what does our classifier call GT vocalizations? ──────────
print(f"\n{'=' * 60}")
print("WHAT DOES CLASSIFIER CALL GT VOCALIZATIONS?")
print("=" * 60)
gt_vv_matched = matched[matched['gt_is_vocalization']]
if len(gt_vv_matched) > 0:
    vv_pred_types = gt_vv_matched['pred_type'].value_counts()
    for t, c in vv_pred_types.items():
        print(f"  {t:20s}: {c:5d} ({c/len(gt_vv_matched)*100:.1f}%)")

# ── 7. What does classifier call GT intrusions? ───────────────────────────
print(f"\n{'=' * 60}")
print("WHAT DOES CLASSIFIER CALL GT INTRUSIONS?")
print("=" * 60)
gt_intr_matched = matched[matched['gt_is_intrusion']]
if len(gt_intr_matched) > 0:
    intr_pred_types = gt_intr_matched['pred_type'].value_counts()
    for t, c in intr_pred_types.items():
        print(f"  {t:20s}: {c:5d} ({c/len(gt_intr_matched)*100:.1f}%)")

# ── 8. Per-session vocalization accuracy ───────────────────────────────────
print(f"\n{'=' * 60}")
print("PER-SESSION VOCALIZATION ACCURACY (top/bottom 5)")
print("=" * 60)
sess_acc = (
    matched.groupby(['subject', 'session'])
    .apply(lambda g: pd.Series({
        'accuracy': ((g['gt_is_vocalization'] == g['pred_is_vocalization']).sum() / len(g)),
        'n_events': len(g),
        'n_gt_vv': g['gt_is_vocalization'].sum(),
    }))
    .reset_index()
    .sort_values('accuracy', ascending=False)
)
print("Best 5:")
print(sess_acc.head(5).to_string(index=False))
print("\nWorst 5:")
print(sess_acc.tail(5).to_string(index=False))
print(f"\nMean accuracy across sessions: {sess_acc['accuracy'].mean():.3f}")
print(f"Median accuracy: {sess_acc['accuracy'].median():.3f}")

# ── 9. Onset timing diff stats ────────────────────────────────────────────
print(f"\n{'=' * 60}")
print("ONSET TIMING DIFFERENCE (matched events)")
print("=" * 60)
onset_diffs = matched['onset_diff_ms']
print(f"  Mean:   {onset_diffs.mean():.0f} ms")
print(f"  Median: {onset_diffs.median():.0f} ms")
print(f"  Std:    {onset_diffs.std():.0f} ms")
print(f"  <500ms: {(onset_diffs < 500).sum()} ({(onset_diffs < 500).mean()*100:.1f}%)")
print(f"  <1000ms: {(onset_diffs < 1000).sum()} ({(onset_diffs < 1000).mean()*100:.1f}%)")

In [ ]:
## ── Error Analysis: Why is vocalization recall low? ────────────────────────

# Many GT vocalizations match to our "Guess" events because the nearest-neighbor
# matching is greedy. Check: are those GT VVs actually close to a word event,
# or are they far away (meaning ASR missed them)?

matched = comparison_df[comparison_df['matched']].copy()
gt_vv_as_guess = matched[(matched.gt_is_vocalization) & (matched.pred_type == 'Guess')]

print("GT vocalizations classified as Guess by our model:")
print(f"  Count: {len(gt_vv_as_guess)}")
print(f"  Onset diff distribution:")
print(f"    Mean:   {gt_vv_as_guess.onset_diff_ms.mean():.0f} ms")
print(f"    Median: {gt_vv_as_guess.onset_diff_ms.median():.0f} ms")
print(f"    >500ms: {(gt_vv_as_guess.onset_diff_ms > 500).sum()} ({(gt_vv_as_guess.onset_diff_ms > 500).mean()*100:.1f}%)")
print(f"    >1000ms: {(gt_vv_as_guess.onset_diff_ms > 1000).sum()} ({(gt_vv_as_guess.onset_diff_ms > 1000).mean()*100:.1f}%)")

# These are cases where the GT has a vocalization but the ASR produced a word
# nearby. The ASR either: (a) didn't detect the vocalization, or (b) transcribed
# the vocalization as a word.
print(f"\n  Top predicted words for these mismatches:")
print(gt_vv_as_guess.pred_word.value_counts().head(20).to_string())

# ── Tighter matching: only count matches within 500ms ─────────────────────
tight = matched[matched.onset_diff_ms <= 500].copy()
gt_vv_t = tight.gt_is_vocalization
pred_vv_t = tight.pred_is_vocalization

tp_t = (gt_vv_t & pred_vv_t).sum()
fp_t = (~gt_vv_t & pred_vv_t).sum()
fn_t = (gt_vv_t & ~pred_vv_t).sum()
tn_t = (~gt_vv_t & ~pred_vv_t).sum()
prec_t = tp_t/(tp_t+fp_t) if tp_t+fp_t else 0
rec_t = tp_t/(tp_t+fn_t) if tp_t+fn_t else 0
f1_t = 2*prec_t*rec_t/(prec_t+rec_t) if prec_t+rec_t else 0

print(f"\n{'='*60}")
print(f"WITH TIGHT MATCHING (<=500ms tolerance)")
print(f"{'='*60}")
print(f"  Matched events: {len(tight)} (of {len(matched)})")
print(f"  TP={tp_t} FP={fp_t} FN={fn_t} TN={tn_t}")
print(f"  Accuracy:  {(tp_t+tn_t)/len(tight):.3f}")
print(f"  Precision: {prec_t:.3f}  Recall: {rec_t:.3f}  F1: {f1_t:.3f}")

# ── Summary interpretation ─────────────────────────────────────────────────
print(f"""
{'='*60}
INTERPRETATION
{'='*60}
The low vocalization recall ({(tp/(tp+fn))*100:.1f}%) is primarily because:
1. The ASR (WhisperX) often does not transcribe vocalizations at all — they
   produce no output word. The GT has a <> event but our pipeline has nothing.
2. When matched greedily, these GT <> events get paired to distant word events,
   inflating false negatives.

With tight matching (<=500ms), recall improves to {rec_t*100:.1f}%, showing that
when the ASR DOES produce output near a GT vocalization, our classifier catches
it {prec_t*100:.1f}% of the time (precision).

Key takeaway: the bottleneck is ASR detection, not classification. Our
VocalizationClassifier correctly identifies {prec_t*100:.0f}% of detected
vocalizations, but WhisperX misses many quiet vocalizations entirely.
""")